Tento projekt načítá data z csv souboru a kontroluje jeho validitu. Pomocí Pandas vyřazuje duplicitní a prázné řádky. Připravuje je na zpracování modelem náhodného klasifikašního lesa (Random Forest Classifier).

Využívám i škálování, které zaručuje konzistenci dat a jejich normalizaci. Bez Scaleru by se souřadnice při různých přiblížení a různě velkých rukou zdály zcela jiné. Je potřeba je škálovat na jednu úroveň, aby obsahovali správné poměřy hodnot.

Model pro Kámen, Nůžky, Papír

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

df = pd.read_csv('gestures_dataset.csv')
print(f"Načteno {len(df)} záznamů.")
df.head()

Načteno 2965 záznamů.


,x0,y0,z0,x1,y1,z1,x2,y2,z2,x3,...,x18,y18,z18,x19,y19,z19,x20,y20,z20,target
0,0.0,0.0,0.0,0.054081,-0.034505,-0.006792,0.091822,-0.103798,-0.005713,0.111053,...,-0.004951,-0.228418,-0.034964,0.002174,-0.176942,-0.023970,-0.011018,-0.145162,-0.009288,0
1,0.0,0.0,0.0,0.056818,-0.039477,-0.007434,0.093618,-0.107245,-0.006953,0.107640,...,-0.009029,-0.231544,-0.032486,-0.000447,-0.179211,-0.021091,-0.013151,-0.143499,-0.005630,0
2,0.0,0.0,0.0,0.055424,-0.041449,-0.009178,0.091439,-0.108773,-0.009747,0.104246,...,-0.012537,-0.232020,-0.032081,-0.003723,-0.180522,-0.020624,-0.015849,-0.145100,-0.004749,0
3,0.0,0.0,0.0,0.055619,-0.038978,-0.010880,0.093028,-0.106184,-0.012023,0.104994,...,-0.011193,-0.229646,-0.033077,-0.002545,-0.178209,-0.022041,-0.014769,-0.142944,-0.006619,0
4,0.0,0.0,0.0,0.055329,-0.038805,-0.010351,0.091976,-0.106484,-0.011285,0.105272,...,-0.010673,-0.230748,-0.033010,-0.002339,-0.178311,-0.021341,-0.014697,-0.143071,-0.005358,0


In [ ]:
# odstranění prázdných hodnot
df_clean = df.dropna()

# rozdělení na vstupy (souřadnice x, y, z) a výstupy (gesto)
X = df_clean.drop('target', axis=1)
y = df_clean['target']

# rozdělení na trénovací a testovací data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# škálování dat
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Trénovací množina: {len(X_train_scaled)} ")

# odstranění duplicitních řádků
df_clean = df.drop_duplicates().reset_index(drop=True)

Data vyčištěna. Trénovací množina: 2372 vzorků.
Nalezeno 0 identických řádků.
Počet řádků po odstranění duplicit: 2965


In [ ]:
# trénování modelu
model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Model natrénován.")

Model byl úspěšně natrénován.


In [1]:
# přesnosti
print(classification_report(y_test, y_pred, target_names=['Rock', 'Paper', 'Scissors', 'Switch']))

NameError: name 'classification_report' is not defined

In [ ]:
# uložení modelu a scaleru pomocí knihovny joblib
joblib.dump(model, 'rps_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

Soubory rps_model.pkl a scaler.pkl byly vyexportovány.


Model pro Abecedu

In [ ]:
import pandas as pd

# načtení dat pro abecedu
df = pd.read_csv('alphabet_dataset.csv')

# smazání prázdmých řádků
df_clean = df.dropna()

# odstranění stejných záznamů
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

In [ ]:
# odstranění výstupu pro trénování
X = df_clean.drop('target', axis=1)
y = df_clean['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# vypsání počtu záznamů
print(df_clean['target'].value_counts())

min_samples = df_clean['target'].value_counts().min()

# undersampling nebo podvzorkování, dosažení stejného počtu záznamů pro každé gesto
df_balanced = df_clean.groupby('target').head(min_samples)

target
25    475
3     468
23    467
21    463
13    449
29    448
18    438
27    435
26    434
19    430
17    429
24    418
22    413
20    410
2     405
28    401
14    398
11    395
9     371
8     371
5     370
1     368
4     361
10    346
12    344
15    332
6     327
0     321
16    318
7     291
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# model, 100 stromů, random_state => počáteční stav gen. random čísel, při každém spuštění se chová stejně
alphabet_model = RandomForestClassifier(n_estimators=100, random_state=42)
alphabet_model.fit(X_train_scaled, y_train)

# přesnost
print(f"Přesnost modelu: {accuracy * 100:.2f}%")
print(classification_report(y_test, alphabet_model.predict(X_test_scaled)))

Přesnost modelu: 99.82%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        72
           1       0.95      0.99      0.97        82
           2       0.97      1.00      0.99        73
           3       0.99      1.00      0.99        92
           4       0.99      0.97      0.98        79
           5       1.00      1.00      1.00        77
           6       1.00      1.00      1.00        60
           7       1.00      1.00      1.00        60
           8       0.99      1.00      0.99        74
           9       1.00      0.91      0.95        78
          10       0.95      1.00      0.98        83
          11       0.99      0.96      0.97        89
          12       0.96      0.97      0.96        67
          13       1.00      0.91      0.96        94
          14       0.96      1.00      0.98        73
          15       1.00      1.00      1.00        65
          16       0.97      0.97      0.97        68
   

In [ ]:
joblib.dump(alphabet_model, 'alphabet_model.pkl')
joblib.dump(scaler, 'alphabet_scaler.pkl')

['alphabet_scaler.pkl']

Model pro číselný dataset

In [ ]:
df = pd.read_csv('num_dataset.csv')

# odstranění řádků s chybějícími hodnotami
df.dropna(inplace=True)

# rozdělení vstupních a výstupních dat, všechny sloupce kromě posledního
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [ ]:
# 80% trénovací a 20% testovací
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Trénovací vzorky: {len(X_train)}, Testovací vzorky: {len(X_test)}")

# počet záznamů pro každé gesto
print(df['target'].value_counts().sort_index())

Data připravena. Trénovací vzorky: 4464, Testovací vzorky: 1117
target
0     354
1     477
2     493
3     473
4     421
5     443
6     390
7     681
8     463
9     783
10    603
Name: count, dtype: int64


In [ ]:
# model o stovce stromů
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Přesnost modelu: {accuracy * 100:.2f}%")
print("\nDetailní report:")
print(classification_report(y_test, y_pred))

Přesnost modelu: 99.82%

Detailní report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        84
           1       1.00      0.99      0.99        94
           2       0.99      1.00      1.00       104
           3       1.00      1.00      1.00        97
           4       0.99      1.00      0.99        89
           5       1.00      1.00      1.00        80
           6       1.00      1.00      1.00        69
           7       1.00      1.00      1.00       130
           8       1.00      1.00      1.00        80
           9       1.00      1.00      1.00       174
          10       1.00      0.99      1.00       116

    accuracy                           1.00      1117
   macro avg       1.00      1.00      1.00      1117
weighted avg       1.00      1.00      1.00      1117



In [ ]:
joblib.dump(model, 'num_model.pkl')
joblib.dump(scaler, 'num_scaler.pkl')

['num_scaler.pkl']